# میانگین امتیاز کالاها بر اساس نظرات مربوط به گارانتی

In [1]:
import pandas as pd
import re
import html
from collections import Counter
from difflib import SequenceMatcher

In [2]:
Train= "train_data.csv"
Brands= "title_brand.csv"
Chunk= 200_000
Min_reviews= 3

# یافتن کلمات مرتبط و غلط املایی‌ها

In [ ]:
seeds= ["warranty", "guarantee"]
token_re= re.compile(r"[a-z]+")

vocab= Counter()
for chunk in pd.read_csv(Train, usecols= ["reviewText", "summary"], chunksize= Chunk):
    text= chunk["reviewText"].fillna("") + " " + chunk["summary"].fillna("")
    for t in text:
        vocab.update(token_re.findall(t.lower()))

rows= []
for w, c in vocab.items():
    if c < 3 or len(w) < 5:
        continue
    sim= max(SequenceMatcher(None, w, s).ratio() for s in seeds)
    if sim >= 0.75:
        rows.append((w, c, round(sim, 2)))

cand_df= (pd.DataFrame(rows, columns= ["word", "count", "similarity"])
             .sort_values("similarity", ascending= False))
print(cand_df[cand_df.similarity >= 0.8].to_string(index= False))

        word  count  similarity
    warranty  23677        1.00
   guarantee   1802        1.00
  guaranteed    693        0.95
  guarantees    155        0.95
  guarrantee      4        0.95
   warrranty      3        0.94
   warrantly      6        0.94
    gurantee      5        0.94
     warraty      5        0.93
     warrany     12        0.93
     waranty     39        0.93
     warrant    463        0.93
   guarnteed      3        0.89
   guarentee     12        0.89
   gaurantee      6        0.89
    watranty      3        0.88
    warrants    143        0.88
    warrenty    131        0.88
guaranteeing     21        0.86
 warrantying      3        0.84
  guarenteed      6        0.84
  gaurantees      4        0.84
  guaranties      3        0.84
  gauranteed      3        0.84
    warantee     17        0.82
   warranted    258        0.82
   warrently      4        0.82
   warrantee    265        0.82
    guaranty     28        0.82
     warenty      7        0.80
     war

به دلیل مشکوک بودن برخی کلمات آنها را بازبینی کرده:

In [4]:
doubtful= ["warrant", "warrants", "warranted", "warrantee", "warantee", "warrent", "warrently"]
print({w: vocab[w] for w in doubtful})

for w in doubtful:
    pat= re.compile(r"(.{40}\b" + w + r"\b.{40})", re.I)
    found= []
    for chunk in pd.read_csv(Train, usecols= ["reviewText"], chunksize= Chunk):
        found += chunk["reviewText"].fillna("").str.extract(pat)[0].dropna().tolist()
        if len(found) >= 8:
            break
    print(f"\n=== {w}")
    for s in found[:8]:
        print("-", s.replace("\n", " "))

{'warrant': 463, 'warrants': 143, 'warranted': 258, 'warrantee': 265, 'warantee': 17, 'warrent': 4, 'warrently': 4}

=== warrant
-  enough of a difference, personally, to warrant the extra money on the more expensive o
- e listed specs and in my opinion do not warrant the removal of any stars.  It would be 
-  the left side went out.  Sony needs to warrant these for over 90 days...if they really
-  both cheap plastic and this should not warrant another $200. I would recommend buying 
- tter with their headphones, these might warrant a four star rating. For everyone else, 
- nas. The first failed shortly after the warrant ended. This one failed after nine month
- wish I took records seriously enough to warrant the overhead this TT requires - there's
- athed the adapter cable in foam wrap to warrant against the cable rattling inside the d

=== warrants
- views ever, but this product definitely warrants one.  It's absolutely fantastic! I'm a 
-  Lifetime Insurance in which Tripp Lite warr

# انتخاب نهایی کلمات

In [5]:
keywords= {"warranty", "warranties", "warrantying", "warrenty", "waranty", "warenty", "warraty", 
            "warrany", "watranty", "warrantly", "warrranty", "guarantee", "guarantees", 
            "guaranteed", "guaranteeing", "guaranties", "guaranty", "guarrantee", "gurantee", 
            "guarentee", "gaurantee", "gaurantees", "guarenteed", "gauranteed", "guarnteed"}

extra_keep= {"warrantee"}
keywords= sorted(keywords | extra_keep)
print(len(keywords), keywords)

pat_final= re.compile(r"\b(?:" + "|".join(map(re.escape, keywords)) + r")\b", re.I)
pat_with= re.compile(r"\b(?:" + "|".join(map(re.escape, keywords + ["warrant", "warrants"])) + r")\b", re.I)

26 ['gaurantee', 'gauranteed', 'gaurantees', 'guarantee', 'guaranteed', 'guaranteeing', 'guarantees', 'guaranties', 'guaranty', 'guarentee', 'guarenteed', 'guarnteed', 'guarrantee', 'gurantee', 'waranty', 'warenty', 'warrantee', 'warranties', 'warrantly', 'warranty', 'warrantying', 'warrany', 'warraty', 'warrenty', 'warrranty', 'watranty']


فیلتر نظرات

In [6]:
usecols= ["asin", "overall", "reviewText", "summary"]
parts, n_total, sum_all= [], 0, 0.0

for chunk in pd.read_csv(Train, usecols= usecols, dtype= {"asin": str}, chunksize= Chunk):
    n_total += len(chunk)
    sum_all += chunk["overall"].sum()
    text= chunk["reviewText"].fillna("") + " " + chunk["summary"].fillna("")
    m_with= text.str.contains(pat_with)
    if not m_with.any():
        continue
    sub= chunk.loc[m_with, ["asin", "overall"]].copy()
    sub["in_final"]= text[m_with].str.contains(pat_final).values
    parts.append(sub)

hits= pd.concat(parts, ignore_index=True)
final_hits= hits[hits.in_final]
extra_hits= hits[~hits.in_final]

اثر حذف warrant*

In [7]:
summary= pd.DataFrame({
    "تعداد نظر": [n_total, len(final_hits), len(hits), len(extra_hits)],
    "میانگین overall": [sum_all / n_total, final_hits.overall.mean(),
                        hits.overall.mean(), extra_hits.overall.mean()],
}, index= ["کل train", "گارانتی‌دار (نهایی)", "با warrant*", "فقط اضافه‌شده‌ها"])
summary.round(3)

,تعداد نظر,میانگین overall
کل train,838944,4.021
گارانتی‌دار (نهایی),18609,3.196
با warrant*,19136,3.213
فقط اضافه‌شده‌ها,527,3.831


# میانگین امتیاز هر کالا

In [ ]:
res= (final_hits.groupby("asin")["overall"]
                 .agg(avg_warranty_rating= "mean", n_reviews= "count")
                 .reset_index())

tb= pd.read_csv(Brands, dtype= {"asin": str}).drop_duplicates("asin")
tb["title"]= tb["title"].fillna("").map(html.unescape)
res= res.merge(tb, on= "asin", how= "left")

print("تعداد کالا با حداقل k نظر:")
for k in (1, 2, 3, 5, 10):
    print(f"  >= {k}: {(res.n_reviews >= k).sum()}")

result= (res[res.n_reviews >= Min_reviews]
          .sort_values(["avg_warranty_rating", "n_reviews"], ascending= [False, False])
          .reset_index(drop=True))
result["avg_warranty_rating"]= result["avg_warranty_rating"].round(3)

result.to_csv("warranty_avg_rating.csv", index=False)
res.to_csv("warranty_avg_rating_all_products.csv", index=False)
print(f"\nکالاهای نهایی (>= {Min_reviews} نظر): {len(result)}")
cols= ["asin", "title", "avg_warranty_rating", "n_reviews"]
display(result.head(15)[cols])
display(result.tail(15)[cols])

تعداد کالا با حداقل k نظر:
  >= 1: 10074
  >= 2: 3098
  >= 3: 1530
  >= 5: 597
  >= 10: 176

کالاهای نهایی (>= 3 نظر): 1530


,asin,title,avg_warranty_rating,n_reviews
0,B00N1O1NQW,iPhone Charger Syncwire Lightning Cable - [App...,5.0,29
1,B00EKJQM5E,"EVGA SuperNOVA 1000 P2, 80+ PLATINUM 1000W, Fu...",5.0,9
2,B00650P2ZC,"Noctua NF-F12 PWM, 4-Pin Premium Quiet Fan (12...",5.0,7
3,B015PC7R6W,Syncwire Lightning to USB A Cable - Nylon Brai...,5.0,7
4,B0002SKHIK,Pelican 1510 Case With Foam (Black),5.0,6
5,B000WU2LXC,"ARCTIC MX-2 - Thermal Compound Paste, Carbon B...",5.0,6
6,B00PK00WRG,82mm X2 UV Filter For Camera Lenses - UV Prote...,5.0,6
7,B00EU799O2,"Macbook Pro 15 Sleeve, Snugg - Brown Leather S...",5.0,5
8,B00O0KISQE,Anker USB 3.0 4-Port Portable Aluminum Hub wit...,5.0,5
9,B00RNGOBYK,amFilm iPad 9.7 Screen Protector Tempered Glas...,5.0,5


,asin,title,avg_warranty_rating,n_reviews
1515,B00INNP5VU,Roku Streaming Stick (3500R) (2014 Model),1.0,3
1516,B00JG01M2U,PNY Turbo Attaché 64GB USB 3.0 Flash Drive - P...,1.0,3
1517,B00L7FLCHM,Toshiba CB35-B3340 13.3 Inch Chromebook (Inte...,1.0,3
1518,B00QKTU8U2,GOgroove FlexSMART X5 Bluetooth FM Transmitter...,1.0,3
1519,B00RCYEL7U,FLIR FX Indoor Wi-Fi Wireless 1080P HD Video M...,1.0,3
1520,B00T63YMGK,"Product TitleVIZIO 60"" Class 4K (2160P) Smart ...",1.0,3
1521,B00T85PCKG,Nikon COOLPIX AW130 Waterproof Digital Camera ...,1.0,3
1522,B00TYFCJOM,Sony BDPS5500 3D Blu-Ray Player with Wi-Fi (20...,1.0,3
1523,B00U78GFWW,Sony HTST9 7.1 Sound Bar with Wireless Subwoof...,1.0,3
1524,B00VIMIV6E,VIZIO E65-C3 65-Inch 1080p Smart LED TV (2015 ...,1.0,3
